In [1]:
!pip install faiss-gpu
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
import json
import time
import torch
import gc
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import faiss
import pickle

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/drive/MyDrive
!ls

/content/drive/MyDrive
 20220414_162844.jpg
 20260605_105320.jpg
'25 interns.xlsx'
 ACADEMICS
'AI Guild Code Report.gdoc'
'APPLICATION FOR NA DEPUTY COORDINATOR 2024-25.gdoc'
 ARJUNAAR_CONSULT_FIXED_RESUME.pdf
'ARJUNAAR_CORE_FIXED_RESUME (2).pdf'
 arjunaar@gmail.com_resume.gdoc
 ArjunAAR_IITMadras_9003866112_Data_Scientist.mp4
'ArjunAAR_IITMadras_9003866112_Software_Engineer (1).mp4'
 ArjunAAR_IITMadras_9003866112_Software_Engineer.mp4
 ARJUNAAR_IITM_IDCARD.pdf
 ARJUNAAR_MASTER_RESUME_NA23B006.docx
 ARJUNAAR_MASTER_RESUME_NA23B006.pdf
'ARJUNAAR_NA23B006 (1) (1).pdf'
 ArjunAAR-na23b006-1740364394823-CV.pdf
'ARJUNAAR_NA23B006 (1).pdf'
'ARJUNAAR_NA23B006 (2).pdf'
'ARJUNAAR_NA23B006 (3).pdf'
'ARJUNAAR_NA23B006 (4).pdf'
'ARJUNAAR_NA23B006 (5).pdf'
'ARJUNAAR_NA23B006 (6).pdf'
'ARJUNAAR_NA23B006 (7).pdf'
 ARJUNAAR_NA23B006_ASSIGNMENT1.pdf
 ARJUNAAR_NA23B006_LR1.py
 ARJUNAAR_NA23B006_LR2.py
 ARJUNAAR_NA23B006.pdf
 ARJUNAAR_PASSPORT_PHOTO.JPEG
 ARJUNAAR_RESUME_NA23B006_2.pdf
'ARJUNAAR_RESUME_NA

In [8]:
%cd /content/drive/MyDrive/vector_store
!ls

/content/drive/MyDrive/vector_store
metadata.pkl  naval_index.faiss


In [3]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
LORA_PATH = "/content/drive/MyDrive/Naval_Phi3_LoRA"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
FAISS_PATH = "/content/drive/MyDrive/vector_store/naval_index.faiss"
METADATA_PATH = "/content/drive/MyDrive/vector_store/metadata.pkl"
TOP_K = 5
MAX_NEW_TOKENS = 200

In [7]:
questions = [
    {
        "id": 1,
        "question": "What is the difference between hogging and sagging in ship structures, and what causes each?",
        "ground_truth": "Hogging occurs when a ship's bow and stern are supported by waves while the middle is unsupported, causing the ship to bend upward (concave down). Sagging is the opposite — the midship is supported by a wave crest while the ends are unsupported, causing the ship to bend downward (concave up). Both create longitudinal bending stresses in the hull girder. Hogging puts the deck in compression and keel in tension; sagging puts the deck in tension and keel in compression."
    },
    {
        "id": 2,
        "question": "Explain the purpose of transverse bulkheads in ship hull design. How do they contribute to structural integrity?",
        "ground_truth": "Transverse bulkheads are vertical walls running across the ship's width. They provide structural rigidity to the hull, resist transverse loads, and most critically, create watertight compartments. If the hull is breached, bulkheads limit flooding to one compartment, preventing the ship from sinking. They also support the deck and resist racking stresses."
    },
    {
        "id": 3,
        "question": "What is metacentric height (GM) and why is it critical for ship stability?",
        "ground_truth": "GM is the distance between the ship's centre of gravity (G) and the metacentre (M) — the point where the vertical through the heeled centre of buoyancy meets the ship's centreline. A positive GM means the ship is stable (righting moment). Larger GM = stiffer ship (faster roll, more initial stability). Smaller GM = tender ship (slower roll but less stability margin). Negative GM means capsizing."
    },
    {
        "id": 4,
        "question": "Describe the difference between a ship's sheer and camber. What are their functional purposes?",
        "ground_truth": "Sheer is the longitudinal curvature of the deck — rising from midship toward bow and stern. It increases freeboard at the ends, improves seaworthiness by keeping the bow/stern drier, and provides reserve buoyancy. Camber is the transverse curvature of the deck — curved upward from the sides to the centreline. It sheds water off the deck and provides additional headroom below."
    },
    {
        "id": 5,
        "question": "What is the significance of the block coefficient (Cb) in ship design, and how does it affect resistance and speed?",
        "ground_truth": "Cb is the ratio of the ship's underwater volume to the volume of a rectangular block with the same length, beam, and draft. A high Cb (>0.8) indicates a full-form ship (tankers, bulkers) — more cargo capacity but higher resistance and lower speed. A low Cb (<0.6) indicates a fine-form ship (warships, fast ferries) — less resistance, higher speed, but less payload. Cb directly affects wavemaking resistance."
    },
    {
        "id": 6,
        "question": "What's the difference between longitudinal and transverse framing?",
        "ground_truth": "Longitudinal framing has frames running along the ship's length (fore-aft), supported by widely-spaced transverse frames. It provides better resistance to longitudinal bending stresses and is preferred for larger ships and tankers. Transverse framing has frames running across the ship's width, supported by longitudinal stringers. It's simpler to construct and preferred for smaller vessels. Longitudinal framing is lighter for the same strength but more complex to build."
    },
    {
        "id": 7,
        "question": "Which type of steel is preferred in shipbuilding? Mild or high strength?",
        "ground_truth": "Mild steel (yield ~235-250 MPa) is commonly used for general ship structures due to its ductility, ease of welding, and lower cost. High-tensile steel (yield ~315-390 MPa) is used in high-stress areas (deck, bottom shell, bulkheads) to reduce plate thickness and save weight. In practice, a combination is used — HT steel where weight savings matter, mild steel elsewhere. Over-reliance on HT steel can reduce ductility and fatigue life."
    },
    {
        "id": 8,
        "question": "What is a bulbous bow, where and why is it used?",
        "ground_truth": "A bulbous bow is a protruding bulb below the waterline at the ship's bow. It reduces wavemaking resistance by creating its own wave system that partially cancels the bow wave, lowering the overall wave resistance. It's most effective at a specific Froude number range (Fn ≈ 0.25-0.35) — typically on medium to high-speed vessels like container ships, naval vessels, and passenger ships. Not beneficial for slow, full-form ships like bulk carriers at low speeds."
    },
    {
        "id": 9,
        "question": "Which series is used in propeller power estimation?",
        "ground_truth": "The Wageningen B-series (developed at MARIN, Netherlands) is the most widely used series for propeller power estimation and design. It provides systematic open-water test data for a range of blade numbers, pitch ratios, and expanded area ratios. It allows estimation of thrust, torque, and efficiency for a given propeller geometry and operating condition."
    },
    {
        "id": 10,
        "question": "Before actual calculation of resistance, which type of scaling would you prefer for the prototype?",
        "ground_truth": "Froude's law of similarity (Froude scaling) is preferred before actual resistance calculations. The model and prototype must have the same Froude number (Fn = V/√(gL)) to ensure similar wavemaking patterns. Reynolds number similarity cannot be simultaneously maintained at model scale, so viscous (frictional) resistance is handled separately using ITTC 1957 or 1978 model-ship correlation lines, with a form factor added for hull shape effects."
    }
]

In [10]:
print("Loading RAG components...")
embed_model = SentenceTransformer(EMBED_MODEL, device="cuda")
index = faiss.read_index(FAISS_PATH)
with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)
print(f"FAISS index loaded: {index.ntotal} vectors")
print(f"Loaded {len(questions)} evaluation questions")

Loading RAG components...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index loaded: 19051 vectors
Loaded 10 evaluation questions


In [11]:
#RAG Functions
def retrieve(query, k=TOP_K):
    query_embedding = embed_model.encode(
        [query],
        normalize_embeddings=True
    )
    D, I = index.search(
        np.asarray(query_embedding).astype("float32"),
        k
    )
    retrieved = []
    for idx in I[0]:
        retrieved.append(metadata[idx])
    return retrieved

In [12]:
def build_prompt(question, retrieved):
    context = ""
    for i, item in enumerate(retrieved):
        context += f"""
Reference {i+1}

Source:
{item['source']}

Content:
{item['text']}

--------------------------------------------

"""
    prompt = f"""
You are an expert Naval Architect specializing in Ship Structural Design.

Answer ONLY from the supplied references.

If the references do not contain the answer,
say so explicitly.

Do not invent information.

================ REFERENCES ================

{context}

============================================

Question:

{question}

Provide a detailed engineering explanation.

Answer:
"""
    return prompt

def generate_answer(model, tokenizer, prompt):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]
    chat = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(
        chat,
        return_tensors="pt",
        truncation=True,
        max_length=3800
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.1,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

In [25]:
def unload_model(model):
    model.cpu()
    del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"GPU memory cleared. {torch.cuda.memory_allocated()/1e9:.2f} GB")

Fine Tuned Model

In [14]:
print("\n" + "="*60)
print("PHASE 1: Loading FINE-TUNED Phi-3 Mini (LoRA)")
print("="*60)


PHASE 1: Loading FINE-TUNED Phi-3 Mini (LoRA)


In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

ft_model = PeftModel.from_pretrained(base_model, LORA_PATH)
ft_model.eval()

print("Fine-tuned model loaded. Running evaluation...")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Fine-tuned model loaded. Running evaluation...


In [16]:
ft_results = []
for q in questions:
    print(f"\nQ{q['id']}: {q['question'][:60]}...")
    start = time.time()
    retrieved = retrieve(q["question"])
    prompt = build_prompt(q["question"], retrieved)
    answer = generate_answer(ft_model, tokenizer, prompt)
    elapsed = time.time() - start
    ft_results.append({
        "id": q["id"],
        "question": q["question"],
        "ground_truth": q["ground_truth"],
        "answer": answer,
        "inference_time": round(elapsed, 2)
    })
    print(f"  Done in {elapsed:.2f}s")
    print(f"  Answer: {answer[:100]}...")


Q1: What is the difference between hogging and sagging in ship s...
  Done in 11.87s
  Answer: Sagging occurs when portions of the vessel's structure experience tensile stresses due to flotation ...

Q2: Explain the purpose of transverse bulkheads in ship hull des...
  Done in 4.93s
  Answer: Transverse bulkheads act primarily as internal stiffening diaphragms within the hull girder resistin...

Q3: What is metacentric height (GM) and why is it critical for s...
  Done in 6.23s
  Answer: Metacentric height (GM) represents the vertical distance between the ship's center of gravity (G) an...

Q4: Describe the difference between a ship's sheer and camber. W...
  Done in 6.54s
  Answer: Camber refers specifically to any continuous convex curvature present throughout the entire extent o...

Q5: What is the significance of the block coefficient (Cb) in sh...
  Done in 5.28s
  Answer: The block coefficient represents the overall volumetric efficiency of the underwater hull shape; low...

Q6:

In [17]:
with open("finetuned_results.json", "w") as f:
    json.dump(ft_results, f, indent=2)
print("\nFine-tuned results saved to finetuned_results.json")


Fine-tuned results saved to finetuned_results.json


In [26]:
unload_model(ft_model)

GPU memory cleared. 0.14 GB


In [27]:
print("\n" + "="*60)
print("PHASE 2: Loading BASE Phi-3 Mini (no LoRA)")
print("="*60)

base_model_only = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
base_model_only.eval()

print("Base model loaded. Running evaluation...")


PHASE 2: Loading BASE Phi-3 Mini (no LoRA)


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Base model loaded. Running evaluation...


In [28]:
base_results = []
for q in questions:
    print(f"\nQ{q['id']}: {q['question'][:60]}...")
    start = time.time()
    retrieved = retrieve(q["question"])
    prompt = build_prompt(q["question"], retrieved)
    answer = generate_answer(base_model_only, tokenizer, prompt)
    elapsed = time.time() - start
    base_results.append({
        "id": q["id"],
        "question": q["question"],
        "ground_truth": q["ground_truth"],
        "answer": answer,
        "inference_time": round(elapsed, 2)
    })
    print(f"  Done in {elapsed:.2f}s")
    print(f"  Answer: {answer[:100]}...")



Q1: What is the difference between hogging and sagging in ship s...
  Done in 10.32s
  Answer: In naval architecture, "hog" refers to the situation where there's more curvature outward along the ...

Q2: Explain the purpose of transverse bulkheads in ship hull des...
  Done in 9.53s
  Answer: Transverse bulkheads serve multiple purposes within the context of naval architecture and play cruci...

Q3: What is metacentric height (GM) and why is it critical for s...
  Done in 9.37s
  Answer: In naval architecture, the term "metacentric height" refers to the vertical distance between the met...

Q4: Describe the difference between a ship's sheer and camber. W...
  Done in 9.54s
  Answer: According to Reference 1 and Refence 2 provided within your document sources, "sheer" refers to the ...

Q5: What is the significance of the block coefficient (Cb) in sh...
  Done in 9.70s
  Answer: The block coefficient (Cb) plays a crucial role in naval architecture concerning ship design because...

Q6:

In [29]:
#Saving the results of base model
with open("base_results.json", "w") as f:
    json.dump(base_results, f, indent=2)
print("\nBase model results saved to base_results.json")


Base model results saved to base_results.json


In [30]:
unload_model(base_model)

GPU memory cleared. 7.79 GB


In [31]:
print("\n\n" + "="*60)
print("COMPARISON: BASE vs FINE-TUNED")
print("="*60)

print(f"\n{'Q#':<4} {'Question':<45} {'Base Time':<12} {'FT Time':<12}")
print("-" * 73)
for base, ft in zip(base_results, ft_results):
    q_short = base['question'][:43]
    print(f"{base['id']:<4} {q_short:<45} {base['inference_time']:.2f}s{'':<6} {ft['inference_time']:.2f}s")

print("\n\nDETAILED ANSWERS:")
print("=" * 80)
for base, ft in zip(base_results, ft_results):
    print(f"\nQ{base['id']}: {base['question']}")
    print(f"\n  GROUND TRUTH:\n  {base['ground_truth'][:250]}...")
    print(f"\n  BASE MODEL ANSWER:\n  {base['answer']}")
    print(f"\n  FINE-TUNED ANSWER:\n  {ft['answer']}")
    print("-" * 80)



COMPARISON: BASE vs FINE-TUNED

Q#   Question                                      Base Time    FT Time     
-------------------------------------------------------------------------
1    What is the difference between hogging and    10.32s       11.87s
2    Explain the purpose of transverse bulkheads   9.53s       4.93s
3    What is metacentric height (GM) and why is    9.37s       6.23s
4    Describe the difference between a ship's sh   9.54s       6.54s
5    What is the significance of the block coeff   9.70s       5.28s
6    What's the difference between longitudinal    9.61s       7.25s
7    Which type of steel is preferred in shipbui   10.03s       7.04s
8    What is a bulbous bow, where and why is it    9.43s       4.64s
9    Which series is used in propeller power est   10.43s       3.59s
10   Before actual calculation of resistance, wh   10.24s       6.45s


DETAILED ANSWERS:

Q1: What is the difference between hogging and sagging in ship structures, and what causes each?

 

In [32]:
output = {
    "base_model_results": base_results,
    "finetuned_model_results": ft_results
}

with open("eval_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("\n\nAll results saved to eval_results.json")
print("Done! Review the answers above and score them manually.")



All results saved to eval_results.json
Done! Review the answers above and score them manually.


## Evaluation

In [34]:
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00


In [8]:
"""
BERTScore Evaluation: Base vs Fine-tuned Phi-3 on Naval Architecture Q&A
No API key needed. Runs on Colab.

Before running:
1. pip install bert-score
2. Place finetuned_results.json and base_results.json in the same directory
"""

import json
from bert_score import score
import numpy as np

# LOAD RESULTS

with open("/content/drive/MyDrive/vector_store/finetuned_results.json", "r") as f:
    ft_results = json.load(f)

with open("/content/drive/MyDrive/vector_store/base_results.json", "r") as f:
    base_results = json.load(f)

print(f"Loaded {len(ft_results)} fine-tuned results and {len(base_results)} base results")


# PREPARE DATA

questions = [item["question"] for item in base_results]
ground_truths = [item["ground_truth"] for item in base_results]
base_answers = [item["answer"] for item in base_results]
ft_answers = [item["answer"] for item in ft_results]


# COMPUTE BERTSCORE

print("\nComputing BERTScore for BASE model answers...")
base_P, base_R, base_F1 = score(
    base_answers,
    ground_truths,
    lang="en",
    model_type="distilbert-base-uncased",
    verbose=True
)

print("\nComputing BERTScore for FINE-TUNED model answers...")
ft_P, ft_R, ft_F1 = score(
    ft_answers,
    ground_truths,
    lang="en",
    model_type="distilbert-base-uncased",
    verbose=True
)


# CONVERT TO LISTS

base_P = base_P.tolist()
base_R = base_R.tolist()
base_F1 = base_F1.tolist()
ft_P = ft_P.tolist()
ft_R = ft_R.tolist()
ft_F1 = ft_F1.tolist()

# COMPARISON TABLE

print("\n\n" + "="*80)
print("BERTScore COMPARISON: BASE vs FINE-TUNED")
print("="*80)
print(f"\n{'Q#':<4} {'Question':<40} {'Base F1':<10} {'FT F1':<10} {'Diff':<10}")
print("-" * 74)

for i in range(len(questions)):
    diff = ft_F1[i] - base_F1[i]
    diff_str = f"+{diff:.4f}" if diff > 0 else f"{diff:.4f}"
    q_short = questions[i][:38]
    print(f"{i+1:<4} {q_short:<40} {base_F1[i]:.4f}{'':<3} {ft_F1[i]:.4f}{'':<3} {diff_str}")

print("-" * 74)

# Averages
avg_base_P = np.mean(base_P)
avg_base_R = np.mean(base_R)
avg_base_F1 = np.mean(base_F1)
avg_ft_P = np.mean(ft_P)
avg_ft_R = np.mean(ft_R)
avg_ft_F1 = np.mean(ft_F1)

print(f"\n{'AVERAGE':<44} {'Base':<10} {'FT':<10} {'Diff':<10}")
print("-" * 74)
print(f"{'Precision':<44} {avg_base_P:.4f}{'':<3} {avg_ft_P:.4f}{'':<3} {'+' + f'{avg_ft_P-avg_base_P:.4f}' if avg_ft_P > avg_base_P else f'{avg_ft_P-avg_base_P:.4f}'}")
print(f"{'Recall':<44} {avg_base_R:.4f}{'':<3} {avg_ft_R:.4f}{'':<3} {'+' + f'{avg_ft_R-avg_base_R:.4f}' if avg_ft_R > avg_base_R else f'{avg_ft_R-avg_base_R:.4f}'}")
print(f"{'F1':<44} {avg_base_F1:.4f}{'':<3} {avg_ft_F1:.4f}{'':<3} {'+' + f'{avg_ft_F1-avg_base_F1:.4f}' if avg_ft_F1 > avg_base_F1 else f'{avg_ft_F1-avg_base_F1:.4f}'}")

improvement = (avg_ft_F1 - avg_base_F1) / avg_base_F1 * 100
print(f"\nF1 Improvement from fine-tuning: +{improvement:.1f}%")
print(f"Base F1: {avg_base_F1:.4f} → Fine-tuned F1: {avg_ft_F1:.4f}")

print("\n\n" + "="*80)
print("DETAILED PER-QUESTION BREAKDOWN")
print("="*80)

for i in range(len(questions)):
    print(f"\nQ{i+1}: {questions[i]}")
    print(f"  Ground Truth: {ground_truths[i][:150]}...")
    print(f"  Base Answer:   {base_answers[i][:150]}...")
    print(f"  FT Answer:     {ft_answers[i][:150]}...")
    print(f"  Base — P: {base_P[i]:.4f}  R: {base_R[i]:.4f}  F1: {base_F1[i]:.4f}")
    print(f"  FT   — P: {ft_P[i]:.4f}  R: {ft_R[i]:.4f}  F1: {ft_F1[i]:.4f}")
    print("-" * 60)

# SAVE RESULTS
output = {
    "base_model": {
        "avg_precision": avg_base_P,
        "avg_recall": avg_base_R,
        "avg_f1": avg_base_F1,
        "per_question": [
            {
                "id": i+1,
                "question": questions[i],
                "precision": base_P[i],
                "recall": base_R[i],
                "f1": base_F1[i]
            }
            for i in range(len(questions))
        ]
    },
    "finetuned_model": {
        "avg_precision": avg_ft_P,
        "avg_recall": avg_ft_R,
        "avg_f1": avg_ft_F1,
        "per_question": [
            {
                "id": i+1,
                "question": questions[i],
                "precision": ft_P[i],
                "recall": ft_R[i],
                "f1": ft_F1[i]
            }
            for i in range(len(questions))
        ]
    },
    "improvement": {
        "f1_absolute": avg_ft_F1 - avg_base_F1,
        "f1_percentage": improvement,
        "base_f1": avg_base_F1,
        "finetuned_f1": avg_ft_F1
    }
}

with open("bertscore_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("\nResults saved to bertscore_results.json")


Loaded 10 fine-tuned results and 10 base results

Computing BERTScore for BASE model answers...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.12 seconds, 80.01 sentences/sec


BERTScore COMPARISON: BASE vs FINE-TUNED

Q#   Question                                 Base F1    FT F1      Diff      
--------------------------------------------------------------------------
1    What is the difference between hogging   0.7868    0.7847    -0.0021
2    Explain the purpose of transverse bulk   0.7864    0.8174    +0.0310
3    What is metacentric height (GM) and wh   0.8077    0.8347    +0.0269
4    Describe the difference between a ship   0.8090    0.8072    -0.0018
5    What is the significance of the block    0.7756    0.7776    +0.0020
6    What's the difference between longitud   0.7766    0.8155    +0.0389
7    Which type of steel is preferred in sh   0.7652    0.7894    +0.0242
8    What is a bulbous bow, where and why i   0.7416    0.8073    +0.0657
9    Which series is used in propeller powe   0.7768    0.7694    -0.0075
10   Before actual calculation of resistanc   0.7694    0.7963    +0.0269
--------------------